# Pertemuan 08 — Regresi Linear (Bagian 3): Sudut Pandang Probabilistik

Pada modul ini kita akan mempelajari **sudut pandang probabilistik** dari regresi linear.

> **Kaitan dengan pertemuan sebelumnya.** Pertemuan 06 memperkenalkan regresi linear dan fungsi basis, Pertemuan 07 membahas geometri dan regularisasi. Sekarang kita melihatnya dari sisi yang ketiga: **statistika**. Ternyata metode kuadrat terkecil yang selama ini kita pakai bukan sekadar pilihan yang kebetulan masuk akal — ia adalah **penduga kemungkinan maksimum** (*maximum likelihood*) di bawah asumsi derau Gaussian.

### Apa yang akan kita kerjakan

| Bagian | Isi |
|---|---|
| 1 | Ketika persamaan normal bermasalah: ill-conditioning |
| 2 | Kuadrat terkecil = kemungkinan maksimum (MLE) |
| 3 | Ketidakpastian dan selang kepercayaan |
| 4 | Teori keputusan: distribusi derau menentukan fungsi kerugian |

**Prasyarat:** regresi linear dan regularisasi (Pertemuan 06–07), serta konsep distribusi Gaussian (Pertemuan 05).


In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as st
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

np.random.seed(42)

import plotly.io as pio
pio.renderers.default = "notebook_connected"

---
## 1. Ketika Persamaan Normal Bermasalah

### 1.1 Ill-Conditioning (Matriks Buruk Kondisi)

Model regresi linear dapat mengalami **ketidakstabilan numerik** ketika matriks rancangannya hampir **berkolinear**, sehingga menghasilkan matriks yang buruk kondisinya (*ill-conditioned*).

**Mengapa bisa terjadi?** Penyelesaian regresi linear menuntut kita membalik $\mathbb{X}^T\mathbb{X}$. Bila dua fitur nyaris identik — misalnya "tinggi badan dalam sentimeter" dan "tinggi badan dalam inci" — maka salah satu kolom hampir merupakan kelipatan kolom lainnya. Matriks itu menjadi **nyaris singular**, dan membaliknya menghasilkan bobot yang sangat besar serta saling meniadakan.

**Bilangan kondisi** mengukur keparahannya:

$$
\kappa(\mathbb{X}) = \frac{\sigma_{\max}}{\sigma_{\min}}
$$

yaitu perbandingan nilai singular terbesar dan terkecil. Karena $\mathbb{X}^T\mathbb{X}$ mengkuadratkan nilai singular, bilangan kondisinya menjadi $\kappa^2$ — masalahnya berlipat ganda.

**Bagaimana Ridge menolong?** Ridge mengganti $\mathbb{X}^T\mathbb{X}$ menjadi $\mathbb{X}^T\mathbb{X} + \lambda\mathbf{I}$. Menambahkan $\lambda$ pada diagonal mengangkat seluruh nilai singular menjauhi nol, sehingga matriksnya kembali dapat dibalik dengan stabil.

Pada percobaan berikut kita sengaja membuat dua fitur yang hampir identik ($x_2 = x_1$ ditambah gangguan sangat kecil), lalu membandingkan bobot hasil persamaan normal biasa dengan bobot hasil Ridge. Bobot sebenarnya adalah $[3, 2, -1]$.


In [ ]:
np.random.seed(0)

# Menyusun matriks rancangan yang hampir berkolinear
N, D = 500, 2                       # satu intersep + 2 fitur
X = np.ones((N, D + 1))             # kolom pertama = intersep
x1 = np.random.randn(N)
x2 = x1 + 1e-4 * np.random.randn(N) # fitur yang hampir identik dengan x1
X[:, 1] = x1
X[:, 2] = x2

t = 3 + 2*x1 - 1*x2 + 0.1*np.random.randn(N)  # bobot sebenarnya: [3, 2, -1]

# 2. Penyelesaian lewat persamaan normal
XtX   = X.T @ X
Xty   = X.T @ t
w_ne  = np.linalg.solve(XtX, Xty)

# 3. Menghitung bilangan kondisi
u, s, vt = np.linalg.svd(X, full_matrices=False)
kappa = s.max() / s.min()

# 4. Penyelesaian lewat Ridge (λ = 1e-2)
lam   = 1e-2
w_ridge = np.linalg.solve(XtX + lam*np.eye(D+1), Xty)

print("Bilangan kondisi cond(XᵀX) =", kappa**2)   # κ² karena XᵀX mengkuadratkan nilai singular
print("Bobot lewat persamaan normal:", w_ne)
print("Bobot lewat Ridge           :", w_ridge)

**Bacalah keluaran di atas.**

* **Bilangan kondisi** bernilai sangat besar — inilah tanda matriks yang buruk kondisinya.
* **Bobot dari persamaan normal** meleset jauh dari nilai sebenarnya $[3, 2, -1]$. Biasanya dua bobot terakhir menjadi sangat besar dengan tanda berlawanan sehingga pengaruhnya saling meniadakan.
* **Bobot dari Ridge** jauh lebih terkendali dan masuk akal.

> **Catatan penting.** Karena $x_1$ dan $x_2$ hampir identik, sesungguhnya tidak ada satu jawaban tunggal yang benar — banyak pasangan bobot menghasilkan prediksi yang sama baiknya. Yang rusak di sini bukan kemampuan **memprediksi**, melainkan kemampuan **menafsirkan** koefisiennya. Ridge memilih di antara jawaban-jawaban itu yang normanya paling kecil, dan itulah pilihan paling stabil.

---
## 2. Kuadrat Terkecil ≘ Kemungkinan Maksimum

Bagian ini menunjukkan bahwa penyelesaian **kuadrat terkecil** (*least squares*) sama dengan **penduga kemungkinan maksimum** (*maximum likelihood estimation*, MLE) di bawah asumsi derau Gaussian. Inilah yang memberi tafsiran probabilistik pada regresi linear, sekaligus menghubungkan optimisasi dengan inferensi statistik.

**Modelnya.** Kita andaikan data dibangkitkan oleh:

$$
\mathbf{y} = \mathbb{X}\mathbf{w}^* + \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(0, \sigma^2 \mathbf{I})
$$

Artinya: ada hubungan linear yang sebenarnya, tetapi setiap pengamatan tercemar derau Gaussian yang saling bebas dengan ragam yang sama.

**Log-likelihood-nya.** Dengan asumsi itu, log-likelihood untuk $\mathbf{w}$ adalah:

$$
\log L(\mathbf{w}) = -\frac{1}{2\sigma^2}\|\mathbf{y} - \mathbb{X}\mathbf{w}\|^2 + \text{konstanta}
$$

**Kesimpulannya.** Memaksimalkan $\log L(\mathbf{w})$ sama persis dengan **meminimalkan** $\|\mathbf{y} - \mathbb{X}\mathbf{w}\|^2$, yaitu jumlah kuadrat galat. Keduanya hanya berbeda oleh faktor pengali positif $\frac{1}{2\sigma^2}$ dan sebuah konstanta — dan keduanya tidak mengubah letak titik minimumnya.

> **Mengapa ini penting?** Selama ini kita memakai kuadrat galat karena "masuk akal". Sekarang kita tahu alasan yang lebih dalam: mengkuadratkan galat adalah konsekuensi langsung dari asumsi bahwa deraunya berdistribusi Gaussian. Pada Bagian 4 kita akan melihat bahwa asumsi derau yang berbeda menghasilkan fungsi kerugian yang berbeda pula.

Sel berikut memeriksa hal itu secara numerik dalam dua bagian: **Bagian A** membandingkan kedua fungsi tujuan pada satu dataset, dan **Bagian B** memeriksa sifat **tak bias** dengan simulasi Monte Carlo atas 2000 dataset.


In [ ]:
# Model linear: y = Xw* + ε, dengan ε ~ N(0, σ²I)
rng = np.random.default_rng(0)
n = 80
x = np.linspace(0, 5, n)
X = np.c_[x, np.ones_like(x)]                 # kolom [x, 1]
w_star = np.array([2.0, 1.0])             # kemiringan dan intersep sebenarnya
sigma = 1.0                                   # simpangan baku derau

def ols_w(X, y):
    # Rumus tertutup kuadrat terkecil — sekaligus MLE untuk w bila deraunya Gaussian
    return np.linalg.pinv(X) @ y

# BAGIAN A: dua fungsi tujuan (SSE vs log-likelihood) pada satu dataset
eps = rng.normal(0, sigma, size=n)
y = X @ w_star + eps
w_hat = ols_w(X, y)

def sse(w):            # jumlah kuadrat galat
    r = y - X @ w
    return float(r @ r)

def neg_loglik(w):     # −log L(w | y) dengan σ diketahui (konstanta diabaikan)
    # −log L = (1/(2σ²)) · ||y − Xw||² + konstanta → titik minimumnya sama dengan SSE
    return 0.5 * sse(w) / (sigma**2)

print("Pada satu dataset:")
print("  ŵ (kuadrat terkecil / MLE) =", w_hat)
print("  SSE pada ŵ                 =", sse(w_hat))
print("  −logL pada ŵ               =", neg_loglik(w_hat))
# Kedua fungsi tujuan hanya berbeda faktor pengali positif dan konstanta → titik minimumnya sama.

# --- BAGIAN B: sifat tak bias, diperiksa dengan simulasi Monte Carlo ---
M = 2000
ws = []
for _ in range(M):
    y_m = X @ w_star + rng.normal(0, sigma, size=n)
    ws.append(ols_w(X, y_m))
ws = np.vstack(ws)
w_bar = ws.mean(axis=0)               # E[ŵ] ≈ w*, inilah sifat tak bias

# Nilai harapan fungsi tujuan: rata-rata SSE(w) atas banyak dataset.
# Titik minimum dari E[SSE(w)] seharusnya tepat di w*.
def avg_sse(w):
    # Memakai kembali y_m yang sama agar tidak perlu dibangkitkan ulang
    Xw = X @ w
    errs = []
    for i in range(M):
        # reconstruct y_m from stored ŵ? Simpler: re-sample with a fixed seed per i
        # but to keep it deterministic and fast, approximate with theory:
        # E[||y - Xw||^2] = ||X(w* - w)||^2 + nσ^2
        pass

**Bacalah keluaran Bagian A.** Nilai SSE dan $-\log L$ pada $\hat{\mathbf{w}}$ berbeda angkanya, tetapi keduanya dicapai pada **titik $\mathbf{w}$ yang sama**. Inilah bukti numerik dari kesetaraan yang dibahas di atas: mengalikan sebuah fungsi dengan bilangan positif dan menambahkan konstanta tidak menggeser letak titik minimumnya.

Sel berikutnya memeriksa Bagian B, yaitu sifat **tak bias** penduga kuadrat terkecil.

In [ ]:
# Secara teori: E[||y - Xw||²] = ||X(w* - w)||² + nσ²  → minimum tepat di w = w*.

# Menghitung nilai harapan SSE pada kisi di sekitar w*, lalu mencari titik minimumnya.
slopes = np.linspace(w_star[0]-1.0, w_star[0]+1.0, 101)
ints   = np.linspace(w_star[1]-1.0, w_star[1]+1.0, 101)

def expected_sse(w):
    d = w_star - w
    return float(np.linalg.norm(X @ d)**2 + n * sigma**2)

E = np.array([[expected_sse(np.array([a,b])) for b in ints] for a in slopes])
i, j = np.unravel_index(E.argmin(), E.shape)
w_argmin_expected = np.array([slopes[i], ints[j]])

print("\nPemeriksaan pada tingkat nilai harapan:")
print("  w* (nilai sebenarnya)        =", w_star)
print("  rata-rata w-topi atas 2000 dataset =", w_bar, "(≈ w*)")
print("  argmin E[SSE(w)]              =", w_argmin_expected, "(= w* pada kisi)")

**Bacalah ketiga baris keluaran di atas.**

* Baris pertama adalah nilai sebenarnya $\mathbf{w}^* = [2, 1]$.
* Baris kedua adalah rata-rata $\hat{\mathbf{w}}$ atas 2000 dataset berbeda. Nilainya sangat mendekati $\mathbf{w}^*$ — inilah arti **penduga tak bias**: pada setiap dataset tunggal hasilnya meleset, tetapi rata-ratanya tepat.
* Baris ketiga adalah titik yang meminimalkan **nilai harapan** SSE, yang juga jatuh tepat di $\mathbf{w}^*$.

> **Perhatikan suku $n\sigma^2$** pada rumus $E[\|y - Xw\|^2] = \|X(w^* - w)\|^2 + n\sigma^2$. Suku itu **tidak bergantung pada $\mathbf{w}$**, sehingga tidak dapat dihilangkan oleh pilihan bobot mana pun. Inilah **galat tak tereduksi** (*irreducible error*): batas bawah kesalahan yang tidak dapat ditembus model sebaik apa pun, karena berasal dari derau itu sendiri.

---
## 3. Ketidakpastian dan Selang Kepercayaan

Memahami **ketidakpastian** pada prediksi sangat penting untuk menilai keandalan model. Bagian ini memperagakan cara menghitung selang kepercayaan (*confidence interval*), baik untuk satu titik pengamatan maupun untuk rata-rata beberapa pengamatan.

**Dua jenis ketidakpastian yang berbeda:**

| Jenis | Rumus lebarnya | Menjawab pertanyaan |
|---|---|---|
| Satu pengamatan baru | $1{,}96\,\sigma$ | Seberapa jauh **satu** nilai baru bisa menyimpang dari prediksi? |
| Rata-rata $n$ pengamatan | $1{,}96\,\sigma/\sqrt{n}$ | Seberapa jauh **rata-rata** $n$ nilai baru bisa menyimpang? |

**Mengapa yang kedua lebih sempit?** Karena merata-ratakan meredam derau. Semakin banyak pengamatan yang dirata-ratakan, semakin kecil ragamnya — mengikuti faktor $1/\sqrt{n}$. Inilah alasan pengukuran berulang lebih dapat dipercaya daripada satu kali pengukuran.

> **Angka 1,96** berasal dari distribusi normal baku: sekitar 95% massa peluangnya berada dalam rentang $\pm 1{,}96$ simpangan baku dari rata-rata.

Pada percobaan berikut kita membangkitkan data sinus berderau, mencocokkan polinomial derajat tiga, lalu menggambar kedua pita kepercayaan itu.


In [ ]:
# 1. Data buatan: gelombang sinus berderau
np.random.seed(42)
N = 80
x = np.random.uniform(-1, 1, (N, 1))
σ = 0.25
t = np.sin(3*np.pi*x) + σ*np.random.randn(N, 1)

# 2. Mencocokkan polinomial derajat tiga
Φ = PolynomialFeatures(3, include_bias=True)
X = Φ.fit_transform(x)
w = np.linalg.lstsq(X, t, rcond=None)[0]

# 3. Kisi rapat untuk menggambar kurva prediksi
xg = np.linspace(-1, 1, 300)[:, None]
Xg = Φ.transform(xg)
y_pred = (Xg @ w).ravel()

# 4. Pita kepercayaan 95%
ci_single = 1.96 * σ                     # untuk SATU pengamatan baru
ci_mean10 = 1.96 * σ / np.sqrt(10)       # untuk RATA-RATA 10 pengamatan

# 5. Menggambar hasilnya
plt.figure(figsize=(6,4))
plt.scatter(x, t, label="Data", alpha=.5)
plt.plot(xg, y_pred, color="#D55E00", lw=2, label="Prediksi $y(x,w)$")
plt.fill_between(xg.ravel(),
                 y_pred - ci_single,
                 y_pred + ci_single,
                 color="#0072B2", alpha=.25, label="Selang 95% (satu pengamatan)")
plt.fill_between(xg.ravel(),
                 y_pred - ci_mean10,
                 y_pred + ci_mean10,
                 color="#009E73", alpha=.25, label="Selang 95% (rata-rata 10)")
plt.xlabel("$x$"); plt.ylabel("$t$")
plt.title("Pita Kepercayaan Menggambarkan Ketidakpastian Prediksi")
plt.legend(fontsize=9); plt.grid(alpha=.3); plt.tight_layout();
plt.show()


**Bacalah grafik di atas.**

Pita biru (lebih lebar) menyatakan ketidakpastian bagi **satu pengamatan baru**. Bila besok Anda mengukur satu nilai $t$ pada suatu $x$, kira-kira 95% kemungkinannya nilai itu jatuh di dalam pita biru.

Pita hijau (lebih sempit) menyatakan ketidakpastian bagi **rata-rata 10 pengamatan**. Pita ini $\sqrt{10} \approx 3{,}16$ kali lebih sempit.

> **Jangan tertukar.** Pita yang lebih sempit **bukan** berarti modelnya lebih baik. Keduanya menjawab pertanyaan yang berbeda. Kesalahan yang sering terjadi dalam laporan adalah memakai pita rata-rata untuk menjanjikan ketepatan pada satu pengamatan tunggal.

---
## 4. Teori Keputusan

**Teori keputusan** memberi kerangka untuk mengambil keputusan optimal di bawah ketidakpastian. Bagian ini memperkenalkan beberapa distribusi derau (Gaussian dan Laplace) beserta fungsi kerugian yang bersesuaian (MSE dan MAE), lalu menunjukkan bagaimana pilihan itu memengaruhi **ketangguhan** (*robustness*) model regresi.

**Hubungan derau dengan fungsi kerugian.** Melanjutkan gagasan Bagian 2, fungsi kerugian sesungguhnya adalah **negatif log-likelihood** dari asumsi derau kita:

| Asumsi derau | PDF-nya sebanding dengan | $-\log p$ menghasilkan | Nama kerugiannya |
|---|---|---|---|
| Gaussian | $e^{-\varepsilon^2/2}$ | $\varepsilon^2$ | **MSE** (kuadrat galat) |
| Laplace | $e^{-\lvert\varepsilon\rvert}$ | $\lvert\varepsilon\rvert$ | **MAE** (galat mutlak) |

**Akibat praktisnya.** Perhatikan bentuk ekor kedua distribusi pada grafik berikut. Laplace memiliki **ekor yang lebih tebal**, artinya ia menganggap galat besar sebagai sesuatu yang lebih wajar terjadi. Karena itu:

* **MSE** menghukum galat besar secara kuadratik, sehingga sangat **peka terhadap pencilan** (*outlier*). Satu titik yang jauh menyimpang dapat menarik seluruh garis regresi.
* **MAE** menghukum secara linear, sehingga jauh **lebih tangguh terhadap pencilan**.

> **Kesimpulan praktis.** Memilih fungsi kerugian bukan sekadar urusan matematis, melainkan pernyataan tentang **jenis derau apa yang Anda yakini ada pada data Anda**. Bila data Anda mengandung pencilan, MAE atau Huber loss sering lebih tepat daripada MSE.


In [ ]:
ε = np.linspace(-4, 4, 400)

# Fungsi kepadatan peluang kedua distribusi derau
gauss = st.norm(0, 1).pdf(ε)
lap   = st.laplace(0, 1/np.sqrt(2)).pdf(ε)      # ragamnya disamakan dengan N(0,1)


# Fungsi kerugian, yaitu −log p sampai suatu konstanta
rss   = ε**2
mae   = np.abs(ε)

fig, ax = plt.subplots(1, 2, figsize=(9,4))
ax[0].plot(ε, gauss, label='Gaussian'); ax[0].plot(ε, lap, label='Laplace')
ax[0].set_title("Kepadatan Peluang Derau")
ax[0].legend(); ax[0].set_ylim(0,1)

ax[1].plot(ε, rss,   label='MSE');   ax[1].plot(ε, mae, label='MAE')
ax[1].set_title("Fungsi Kerugian yang Bersesuaian")
ax[1].legend(); ax[1].set_ylim(0,6)

plt.tight_layout()
plt.show()


**Bacalah kedua panel.**

Panel kiri menampilkan kedua distribusi derau dengan **ragam yang sama**. Perhatikan bahwa Laplace lebih runcing di tengah tetapi **ekornya lebih tebal** — ia menganggap galat besar lebih mungkin terjadi daripada anggapan Gaussian.

Panel kanan menampilkan fungsi kerugian yang bersesuaian. Perhatikan bahwa MSE (parabola) tumbuh jauh lebih cepat daripada MAE (garis berbentuk V) ketika galatnya membesar.

> **Akibat praktisnya.** Bayangkan satu titik data yang menyimpang sejauh 4 satuan. MSE memberinya hukuman 16, sedangkan MAE hanya 4. Karena hukumannya jauh lebih besar, model yang dilatih dengan MSE akan **menggeser seluruh garis regresi** demi mengurangi galat pada satu titik itu. Itulah sebabnya MSE peka terhadap pencilan, sementara MAE jauh lebih tangguh.

---
## 5. Penutup

### Ringkasan

**Ill-conditioning**

* Fitur yang hampir berkolinear membuat $\mathbb{X}^T\mathbb{X}$ nyaris singular, sehingga bobotnya meledak dan sangat peka terhadap perubahan kecil pada data.
* Bilangan kondisi $\kappa = \sigma_{\max}/\sigma_{\min}$ mengukur keparahannya; untuk $\mathbb{X}^T\mathbb{X}$ nilainya menjadi $\kappa^2$.
* Ridge menambahkan $\lambda\mathbf{I}$ pada diagonal sehingga matriksnya kembali stabil.

**Kuadrat terkecil = MLE**

* Di bawah asumsi derau Gaussian, meminimalkan jumlah kuadrat galat **setara** dengan memaksimalkan likelihood.
* Kedua fungsi tujuan hanya berbeda oleh faktor pengali positif dan konstanta, sehingga titik minimumnya sama.
* Penduga kuadrat terkecil bersifat **tak bias**: rata-rata $\hat{\mathbf{w}}$ atas banyak dataset menuju $\mathbf{w}^*$.

**Ketidakpastian**

* Selang kepercayaan untuk satu pengamatan baru selebar $1{,}96\sigma$; untuk rata-rata $n$ pengamatan menjadi $1{,}96\sigma/\sqrt{n}$.
* Merata-ratakan meredam derau mengikuti faktor $1/\sqrt{n}$.

**Teori keputusan**

* Fungsi kerugian adalah negatif log-likelihood dari asumsi derau kita.
* Derau Gaussian → MSE (peka terhadap pencilan); derau Laplace → MAE (tangguh terhadap pencilan).

### Latihan Mandiri

1. Pada Bagian 1, ubah gangguan `1e-4` menjadi `1e-1` sehingga kedua fitur tidak lagi hampir identik. Bagaimana bilangan kondisinya berubah? Apakah bobot dari persamaan normal menjadi wajar kembali?
2. Masih di Bagian 1, coba beberapa nilai `lam`: `1e-6`, `1e-2`, dan `1`. Berapa nilai yang menghasilkan bobot paling dekat dengan nilai sebenarnya $[3, 2, -1]$?
3. Pada Bagian 2, ubah `sigma` dari 1,0 menjadi 0,1 lalu 3,0. Bagaimana pengaruhnya terhadap sebaran $\hat{\mathbf{w}}$ pada simulasi Monte Carlo? Apakah rata-ratanya tetap mendekati $\mathbf{w}^*$?
4. Pada Bagian 2, naikkan `M` dari 2000 menjadi 10000. Apakah `mean(ŵ)` menjadi lebih dekat ke `w*`? Jelaskan kaitannya dengan hukum bilangan besar.
5. Pada Bagian 3, ubah `ci_mean10` menjadi rata-rata 100 pengamatan. Berapa kali lipat pita itu menyempit dibandingkan pita satu pengamatan?
6. Pada Bagian 3, naikkan derajat polinomial dari 3 menjadi 15. Apakah pita kepercayaan masih menggambarkan ketidakpastian dengan jujur? Kaitkan dengan overfitting.
7. Tambahkan satu pencilan ekstrem pada data Bagian 3 (misalnya $t = 10$ pada suatu $x$), lalu cocokkan ulang. Seberapa jauh kurvanya tertarik? Menurut Anda apa yang akan terjadi bila memakai MAE alih-alih MSE?
8. Buktikan dengan kalimat Anda sendiri: mengapa memaksimalkan $\log L(\mathbf{w})$ menghasilkan titik yang sama dengan meminimalkan $\|\mathbf{y} - \mathbb{X}\mathbf{w}\|^2$?

### Bacaan Lanjutan

* Regresi tangguh terhadap pencilan: https://scikit-learn.org/stable/modules/linear_model.html#robustness-regression
* `HuberRegressor` — kompromi antara MSE dan MAE
